# NyquistGPU web UI on the Colab GPU

The same page as https://bachrathyd.github.io/InterpolatedNyquist.jl/webgpu/ — equation text,
sliders, progressive charts up to the screen resolution, zoom, PNG export — but the 2-D marches
run here, on the Colab GPU, with NyquistGPU (CUDA): Float32 for the coarse levels while dragging,
**Float64 for the final level**. The colouring stays on your browser's WebGPU (use Chrome / Edge).
3-D views and `integral(...)` by quadrature are computed in the browser as before.

1. *Runtime → Change runtime type → GPU* (T4 is enough to start).
2. Run the cells in order. The first start installs Julia and compiles the kernels (5–10 min);
   every new equation compiles once (some seconds).
3. When done: *Runtime → Disconnect and delete runtime* (compute units).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
%%bash
export PATH=/root/.juliaup/bin:$PATH
if ! command -v julia >/dev/null; then
  curl -fsSL https://install.julialang.org | sh -s -- --yes --default-channel 1.12 > /content/juliaup.log 2>&1
fi
if [ -d /content/InterpolatedNyquist.jl/.git ]; then
  cd /content/InterpolatedNyquist.jl && git fetch -q --depth 1 origin webui-colab && git reset -q --hard FETCH_HEAD
else
  git clone -q -b webui-colab --depth 1 https://github.com/bachrathyd/InterpolatedNyquist.jl /content/InterpolatedNyquist.jl
fi
cd /content/InterpolatedNyquist.jl && git log -1 --format='code: %h %s'
julia --project=gpu/webui -e 'using Pkg; Pkg.instantiate(); Pkg.precompile()' > /content/nyq_webui_setup.log 2>&1
tail -n 3 /content/nyq_webui_setup.log
echo "setup done"

In [ ]:
# start the server (compiles the march kernels before it says "ready")
import subprocess, time, os
os.environ['PATH'] = '/root/.juliaup/bin:' + os.environ['PATH']
log = open('/content/nyq_webui.log', 'w')
srv = subprocess.Popen(['julia', '-t', 'auto', '--project=gpu/webui', 'gpu/webui/server.jl', '8800'],
                       cwd='/content/InterpolatedNyquist.jl', stdout=log, stderr=subprocess.STDOUT)
t0 = time.time()
while True:
    time.sleep(5)
    txt = open('/content/nyq_webui.log', encoding='utf-8', errors='replace').read()
    if 'ready' in txt or srv.poll() is not None or time.time() - t0 > 1800:
        break
    print(f'\r{time.time() - t0:5.0f} s: ' + (txt.strip().splitlines() or ['starting'])[-1][:100], end='')
print()
print(txt[-1500:])

In [ ]:
# the page, with the march on this runtime's GPU
from google.colab import output
output.serve_kernel_port_as_iframe(8800, height=1150)

Server log: `!tail -n 30 /content/nyq_webui.log`. Stop: `srv.terminate()`. The page detects the server
(`api/info`); its device is shown in the header.